# Correlation Analysis: Logical Error Distribution vs Fixed-Class Decoding LLRs

This notebook analyzes the correlation between:
1. **Logical error probability** (from pre-computed distribution)
2. **Correction weight (LLR)** from fixed-class decoding

**Hypothesis**: If the distribution-based gap proxy methods work as intended, logical errors with higher probability should produce lower LLRs when used for fixed-class decoding.

## 1. Run Data Collection Script

First, run the data collection script from the terminal. This separates the long-running simulation from the interactive analysis.

```bash
# Navigate to project root
cd /path/to/ldpc-post-selection

# Run with parallel processing (recommended)
python simulations/analysis/scripts/run_distribution_correlation_analysis.py \
    --n-qubits 144 \
    --p 0.003 \
    --n-shots 10000 \
    --n-errors 10 \
    --n-jobs -1 \
    --output simulations/data/correlation_analysis/bb_n144_p0.003_shots10k.parquet
```

**Options:**
- `--n-qubits, -n`: BB code variant (72, 90, 108, 144, 288, 360, 756)
- `--p`: Physical error rate
- `--n-shots`: Number of shots to analyze (each requires 11 decoder calls)
- `--n-errors`: Number of representative errors to sample from distribution
- `--n-jobs, -j`: Number of parallel jobs (`-1` for all CPUs, default: `1`)
- `--output, -o`: Output path for results (parquet file)

Run `python simulations/analysis/scripts/run_distribution_correlation_analysis.py --help` for all options.

In [ ]:
# Autoreload extension for Jupyter notebooks
%load_ext autoreload
%autoreload 2

# Standard library imports
import json
import os
import sys
from pathlib import Path

# Third-party library imports
import numpy as np
import pandas as pd
from scipy import stats

# Plotly for interactive visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add the parent directory to sys.path
sys.path.append(str(Path(os.getcwd()).parent.parent))

# Pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.6f}'.format)

## 2. Load Results

In [ ]:
# ==============================================================================
# CONFIGURATION - Point to your results file
# ==============================================================================

RESULTS_PATH = Path("../../data/correlation_analysis/bb_n144_p0.003_shots10k.parquet")

# ==============================================================================
# Load results and metadata
# ==============================================================================

if not RESULTS_PATH.exists():
    print(f"Results file not found: {RESULTS_PATH.resolve()}")
    print("\nPlease run the data collection script first:")
    print("")
    print("  python simulations/analysis/scripts/run_distribution_correlation_analysis.py \\")
    print("      --n-qubits 144 --p 0.003 --n-shots 10000 --n-jobs -1 \\")
    print(f"      --output {RESULTS_PATH}")
    raise FileNotFoundError(f"Results not found at {RESULTS_PATH}")

# Load DataFrame
df_results = pd.read_parquet(RESULTS_PATH)

# Load metadata
metadata_path = RESULTS_PATH.with_suffix(".json")
if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)
else:
    metadata = {}

# Display summary
print(f"Loaded {len(df_results):,} records from {RESULTS_PATH.name}")
print(f"\nMetadata:")
for key, value in metadata.items():
    if isinstance(value, list) and len(value) > 5:
        print(f"  {key}: [{value[0]}, {value[1]}, ..., {value[-1]}] (len={len(value)})")
    elif isinstance(value, dict):
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value}")

print(f"\nDataFrame preview:")
df_results.head(10)

## 3. Statistical Analysis

In [ ]:
# Compute correlations
print("=" * 60)
print("STATISTICAL ANALYSIS")
print("=" * 60)

# 1. Spearman rank correlation: error_rank vs fixed_llr
spearman_corr, spearman_p = stats.spearmanr(df_results["error_rank"], df_results["fixed_llr"])
print(f"\n1. Spearman Rank Correlation (error_rank vs fixed_llr):")
print(f"   Correlation: {spearman_corr:.4f}")
print(f"   p-value: {spearman_p:.2e}")
print(f"   Interpretation: {'Significant' if spearman_p < 0.05 else 'Not significant'} at alpha=0.05")
if spearman_corr > 0:
    print(f"   Direction: Positive (higher rank = higher LLR, as expected)")
else:
    print(f"   Direction: Negative (unexpected!)")

# 2. Pearson correlation: log(error_prob) vs fixed_llr
# Filter out zero probabilities for log
df_nonzero = df_results[df_results["error_prob"] > 0].copy()
df_nonzero["log_error_prob"] = np.log10(df_nonzero["error_prob"])
pearson_corr, pearson_p = stats.pearsonr(df_nonzero["log_error_prob"], df_nonzero["fixed_llr"])
print(f"\n2. Pearson Correlation (log10(error_prob) vs fixed_llr):")
print(f"   Correlation: {pearson_corr:.4f}")
print(f"   p-value: {pearson_p:.2e}")
print(f"   Interpretation: {'Significant' if pearson_p < 0.05 else 'Not significant'} at alpha=0.05")
if pearson_corr < 0:
    print(f"   Direction: Negative (higher prob = lower LLR, as expected)")
else:
    print(f"   Direction: Positive (unexpected!)")

# 3. Kruskal-Wallis test: Are LLR distributions different across rank groups?
groups = [group["fixed_llr"].values for _, group in df_results.groupby("error_rank")]
kruskal_stat, kruskal_p = stats.kruskal(*groups)
print(f"\n3. Kruskal-Wallis Test (LLR distributions across rank groups):")
print(f"   H-statistic: {kruskal_stat:.4f}")
print(f"   p-value: {kruskal_p:.2e}")
print(f"   Interpretation: {'Distributions differ significantly' if kruskal_p < 0.05 else 'No significant difference'}")

# 4. Summary statistics per error rank
print(f"\n4. Summary Statistics by Error Rank:")
summary = df_results.groupby("error_rank").agg({
    "error_prob": "first",
    "fixed_llr": ["mean", "std", "median"],
    "llr_delta": ["mean", "std"],
}).round(4)
summary.columns = ["error_prob", "llr_mean", "llr_std", "llr_median", "delta_mean", "delta_std"]
print(summary.to_string())

## 4. Visualizations

In [ ]:
# Box plot: fixed_llr by error_rank
fig = px.box(
    df_results,
    x="error_rank",
    y="fixed_llr",
    title="Fixed-Class LLR Distribution by Error Rank",
    labels={
        "error_rank": "Error Rank (0=most likely, higher=less likely)",
        "fixed_llr": "Fixed-Class LLR",
    },
)
fig.update_layout(
    xaxis_title="Error Rank (0=most likely)",
    yaxis_title="Fixed-Class LLR",
    showlegend=False,
)
fig.add_annotation(
    text=f"Spearman r = {spearman_corr:.3f} (p = {spearman_p:.2e})",
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    showarrow=False,
    font=dict(size=12),
    bgcolor="white",
)
fig.show()

In [ ]:
# Scatter plot: error_prob vs mean fixed_llr with error bars
summary_for_plot = df_results.groupby(["error_rank", "error_prob"]).agg({
    "fixed_llr": ["mean", "std", "count"]
}).reset_index()
summary_for_plot.columns = ["error_rank", "error_prob", "llr_mean", "llr_std", "count"]
summary_for_plot["llr_sem"] = summary_for_plot["llr_std"] / np.sqrt(summary_for_plot["count"])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=summary_for_plot["error_prob"],
    y=summary_for_plot["llr_mean"],
    error_y=dict(
        type="data",
        array=summary_for_plot["llr_sem"] * 1.96,  # 95% CI
        visible=True,
    ),
    mode="markers",
    marker=dict(size=10),
    text=[f"Rank {r}" for r in summary_for_plot["error_rank"]],
    hovertemplate="Prob: %{x:.2e}<br>Mean LLR: %{y:.2f}<br>%{text}<extra></extra>",
))

fig.update_layout(
    title="Error Probability vs Mean Fixed-Class LLR",
    xaxis_title="Error Probability (from distribution)",
    yaxis_title="Mean Fixed-Class LLR",
    xaxis_type="log",
)
fig.add_annotation(
    text=f"Pearson r = {pearson_corr:.3f} (p = {pearson_p:.2e})",
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    showarrow=False,
    font=dict(size=12),
    bgcolor="white",
)
fig.show()

In [ ]:
# Heatmap: shot-level LLRs (subset of shots x errors)
n_shots_total = df_results["shot_id"].nunique()
n_shots_to_show = min(100, n_shots_total)  # Show first 100 shots
df_subset = df_results[df_results["shot_id"] < n_shots_to_show]

pivot_df = df_subset.pivot(index="shot_id", columns="error_rank", values="fixed_llr")

fig = px.imshow(
    pivot_df.values,
    labels=dict(x="Error Rank", y="Shot ID", color="Fixed-Class LLR"),
    x=[f"Rank {r}" for r in pivot_df.columns],
    y=pivot_df.index,
    title=f"Fixed-Class LLR Heatmap (first {n_shots_to_show} shots)",
    color_continuous_scale="Viridis",
    aspect="auto",
)
fig.update_layout(
    xaxis_title="Error Rank (0=most likely)",
    yaxis_title="Shot ID",
)
fig.show()

In [ ]:
# CDF comparison: high-prob errors vs low-prob errors
# Split into top 3 ranks (high prob) vs bottom 3 ranks (low prob)
unique_ranks = sorted(df_results["error_rank"].unique())
n_ranks = len(unique_ranks)
top_ranks = unique_ranks[:3]
bottom_ranks = unique_ranks[-3:]

df_top = df_results[df_results["error_rank"].isin(top_ranks)]
df_bottom = df_results[df_results["error_rank"].isin(bottom_ranks)]

fig = go.Figure()

# Top ranks (high probability errors)
sorted_top = np.sort(df_top["fixed_llr"])
cdf_top = np.arange(1, len(sorted_top) + 1) / len(sorted_top)
fig.add_trace(go.Scatter(
    x=sorted_top,
    y=cdf_top,
    mode="lines",
    name=f"High-prob errors (ranks {list(top_ranks)})",
    line=dict(color="blue"),
))

# Bottom ranks (low probability errors)
sorted_bottom = np.sort(df_bottom["fixed_llr"])
cdf_bottom = np.arange(1, len(sorted_bottom) + 1) / len(sorted_bottom)
fig.add_trace(go.Scatter(
    x=sorted_bottom,
    y=cdf_bottom,
    mode="lines",
    name=f"Low-prob errors (ranks {list(bottom_ranks)})",
    line=dict(color="red"),
))

# Mann-Whitney U test
mw_stat, mw_p = stats.mannwhitneyu(df_top["fixed_llr"], df_bottom["fixed_llr"], alternative="less")

fig.update_layout(
    title="CDF Comparison: High-Probability vs Low-Probability Errors",
    xaxis_title="Fixed-Class LLR",
    yaxis_title="Cumulative Probability",
    legend=dict(x=0.6, y=0.1),
)
fig.add_annotation(
    text=f"Mann-Whitney U: p = {mw_p:.2e}",
    xref="paper", yref="paper",
    x=0.98, y=0.5,
    showarrow=False,
    font=dict(size=12),
    bgcolor="white",
)
fig.show()

print(f"\nMann-Whitney U Test (high-prob < low-prob):")
print(f"  U-statistic: {mw_stat:.1f}")
print(f"  p-value: {mw_p:.2e}")
print(f"  Interpretation: {'High-prob errors have significantly lower LLRs' if mw_p < 0.05 else 'No significant difference'}")

In [ ]:
# Violin plot for more detailed distribution view
fig = px.violin(
    df_results,
    x="error_rank",
    y="fixed_llr",
    box=True,
    points="outliers",
    title="Fixed-Class LLR Distribution by Error Rank (Violin Plot)",
    labels={
        "error_rank": "Error Rank (0=most likely)",
        "fixed_llr": "Fixed-Class LLR",
    },
)
fig.show()

## 5. Conclusions

In [ ]:
print("=" * 60)
print("CONCLUSIONS")
print("=" * 60)

# Summarize findings
print(f"\n1. Spearman Correlation (rank vs LLR): r = {spearman_corr:.4f}")
if abs(spearman_corr) < 0.1:
    correlation_strength = "negligible"
elif abs(spearman_corr) < 0.3:
    correlation_strength = "weak"
elif abs(spearman_corr) < 0.5:
    correlation_strength = "moderate"
else:
    correlation_strength = "strong"
print(f"   Correlation strength: {correlation_strength}")

print(f"\n2. Pearson Correlation (log prob vs LLR): r = {pearson_corr:.4f}")

print(f"\n3. Kruskal-Wallis Test: p = {kruskal_p:.2e}")
print(f"   LLR distributions across ranks: {'significantly different' if kruskal_p < 0.05 else 'not significantly different'}")

print("\n" + "=" * 60)
print("IMPLICATIONS FOR GAP PROXY METHODS")
print("=" * 60)

if abs(spearman_corr) < 0.3 and kruskal_p > 0.05:
    print("""
The analysis suggests WEAK or NO correlation between the logical error
distribution and fixed-class decoding LLRs. This explains why:

- Distribution-based methods (most-likely-first, weighted-random) do not
  outperform random sampling
- The pre-computed distribution does not predict which logical classes
  will have competitive LLRs for a given shot

Possible explanations:
1. The distribution captures global error statistics, not per-shot structure
2. Fixed-class LLRs depend more on the specific syndrome than on prior statistics
3. The distribution may be too coarse-grained to capture relevant correlations
""")
else:
    print("""
The analysis suggests a correlation EXISTS between the logical error
distribution and fixed-class decoding LLRs.

Further investigation needed to understand why distribution-based methods
are not outperforming random sampling despite this correlation.
""")